In [1]:
from tensorflow.keras.datasets import cifar10
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# 載入 CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("訓練資料形狀:", x_train.shape)
print("測試資料形狀:", x_test.shape)




In [ ]:
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy("mixed_float16")

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Current mixed precision policy:", tf.keras.mixed_precision.global_policy())

In [ ]:
from keras.utils import to_categorical
import cv2
# 標籤轉換成 one-hot encoding
y_train = to_categorical(y_train, 10)
y_train = np.concatenate([y_train, y_train], axis=0)

y_test = to_categorical(y_test, 10)


x_train2 = x_train[:, :, ::-1, :]
x_train = np.concatenate([x_train, x_train2], axis=0)

x_train_hsv = np.array([cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32) for img in x_train])
x_train_hsv[:, :, :, 0] /= 179
x_train_hsv[:, :, :, 1] /= 255
x_train_hsv[:, :, :, 2] /= 255

x_test_hsv = np.array([cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32) for img in x_test])
x_test_hsv[:, :, :, 0] /= 179
x_test_hsv[:, :, :, 1] /= 255
x_test_hsv[:, :, :, 2] /= 255


# 正規化圖片像素值到 [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# RGB + HSV
x_train_rgb_hsv = np.concatenate([x_train, x_train_hsv], axis=-1)
x_test_rgb_hsv = np.concatenate([x_test, x_test_hsv], axis=-1)



In [ ]:
print(x_train_rgb_hsv.shape)
print(x_test_rgb_hsv.shape)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Input,Concatenate
from keras import layers, models



def cb(inputs):

    x=layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x=layers.BatchNormalization()(x)
    x=layers.Activation('relu')(x)
    x=layers.Dropout(0.1)(x)
    x=layers.MaxPooling2D((2, 2))(x)

    x=layers.Conv2D(64, (3, 3), padding='same')(x)
    x=layers.BatchNormalization()(x)
    x=layers.Activation('relu')(x)
    x=layers.Dropout(0.1)(x)
    x=layers.MaxPooling2D((2, 2))(x)

    x=layers.Conv2D(128, (3, 3), padding='same')(x)
    x=layers.BatchNormalization()(x)
    x=layers.Activation('relu')(x)

    return x

def testmod(input_shape):
    inputs = Input(input_shape, name="input_layer")
    rgb = inputs[:, :, :, :3]
    hsv = inputs[:, :, :, 3:]

    # 合併
    x = Concatenate()([cb(rgb), cb(hsv)])

    x=layers.Flatten()(x)
    x=layers.Dense(128)(x)
    x=layers.BatchNormalization()(x)
    x=layers.Activation('relu')(x)
    x=layers.Dense(10, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=x)

In [ ]:
m=testmod(input_shape=(32,32,6))

m.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',  # 改成監控 dice monitor='val_dice_coef', val_dice_coef_metric
    factor=0.5,                # 停滯時學習率減半
    patience=8,                # 連續 8 個 epoch 沒改善就降低
    mode='max',                # dice 越大越好
    verbose=0,
    min_lr=1e-6
)


tf_callbackEarlyStopping = tf.keras.callbacks.EarlyStopping(
    patience=20,
    monitor='val_accuracy',
    mode='max',
    restore_best_weights=True,
    verbose=1
)



callbacks=[reduce_lr,tf_callbackEarlyStopping]

In [ ]:

history = m.fit(x_train_rgb_hsv, y_train,
                    epochs=200,
                    batch_size=100,
                    validation_split=0.1,
                    verbose=2,
                    callbacks=callbacks)

In [ ]:
test_loss, test_acc = m.evaluate(x_test_rgb_hsv, y_test, verbose=2)
print("測試集損失:", test_loss)
print("測試集正確率:", test_acc)

In [ ]:
# 繪製 Loss 與 Accuracy 曲線
plt.figure(figsize=(12, 5))

# Loss 曲線
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='訓練 Loss')
plt.plot(history.history['val_loss'], label='驗證 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss 曲線')
plt.legend()

# Accuracy 曲線
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='訓練 Accuracy')
plt.plot(history.history['val_accuracy'], label='驗證 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy 曲線')
plt.legend()

plt.show()